In [1]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT # to create databases
from psycopg2.extras import execute_batch

import pandas as pd

In [3]:
with open('pwd.txt', 'w') as file:
    file.write(input('Enter the password: '))

with open('pwd.txt', 'r') as file:
    pwd = file.read()

In [ ]:
df = pd.read_csv('Flight_data.csv')
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
#print(df.columns.tolist())
#null = df['departure_date'].isnull().sum()
#print(null) #проверили что нулей нет и переводим в тип datatime
df['departure_date'] = pd.to_datetime(df['departure_date'])
#print(df['customer_id'].nunique(), len(df)) # тут проверяем сколько уникальных значений в столбце относительно кол-ва строк, чтобы если длины оказались одинаковыми сделать этот столбец ключом
df_for_sql = df[
    [
        'customer_id',
        'departure_city',
        'arrival_city',
        'departure_date',
        'flight_duration',
        'delay_minutes',
        'name',
        'booking_class',
        'frequent_flyer_status',
        'route',
        'ticket_price',
        'competitor_price',
        'demand',
        'origin',
        'destination',
        'profitability',
        'loyalty_points',
        'churned'
    ]
] # тут поменяла очередность чуть-чуть, чтобы потом customer id в ключи поставить

In [14]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password=pwd,
    host="localhost",
    port="5432"
)

cur = conn.cursor()
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT) 
cur.execute('''
DROP TABLE IF EXISTS flights_raw;
            
CREATE TABLE flights_raw (
    customer_id INT PRIMARY KEY,
    departure_city TEXT,
    arrival_city TEXT,
    departure_date TIMESTAMP,
    flight_duration FLOAT,
    delay_minutes INT,
    name TEXT,
    booking_class TEXT,
    frequent_flyer_status TEXT,
    route TEXT,
    ticket_price FLOAT,
    competitor_price FLOAT,
    demand FLOAT,
    origin TEXT,
    destination TEXT,
    profitability FLOAT,
    loyalty_points INT,
    churned BOOLEAN
);
''')
rows = df_for_sql.values.tolist()
execute_batch(cur, """INSERT INTO flights_raw (customer_id, departure_city,
    arrival_city, departure_date, flight_duration, delay_minutes,
    name, booking_class, frequent_flyer_status, route,
    ticket_price, competitor_price, demand, origin, 
    destination, profitability, loyalty_points, churned) 
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
               """, rows)

In [ ]:
cur.execute("SELECT COUNT(*) FROM flights_raw;")
print(cur.fetchone()) # тут я проверила всё ли у нас загрузилось

(100,)


In [23]:
cur.execute("""
SELECT
    SUM(CASE WHEN departure_city IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN arrival_city IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN departure_date IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN flight_duration IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN delay_minutes IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN booking_class IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN frequent_flyer_status IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN route IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN ticket_price IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN competitor_price IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN demand IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN origin IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN destination IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN profitability IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN loyalty_points IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN churned IS NULL THEN 1 ELSE 0 END)
FROM flights_raw;
""")

print(cur.fetchone()) # в общем получается что в нашей таблице нет NULLs, поэтому и COALESCE(column,0) не пригодилось получается

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)


In [40]:
cur.execute("""
ALTER TABLE flights_raw
ADD COLUMN flight_id INT;
""")

In [ ]:
# cur.execute("""
# WITH ranked AS (
#     SELECT
#         customer_id,
#         DENSE_RANK() OVER (
#             ORDER BY route, origin, destination, departure_date
#         ) AS new_flight_id
#     FROM flights_raw
# )
# UPDATE flights_raw f
# SET flight_id = r.new_flight_id
# FROM ranked r
# WHERE f.customer_id = r.customer_id;
# """)
# если мы делаем так, то у нас вообще все рейсы имеют всой отдельный айди

In [58]:
cur.execute("""
WITH ranked AS (
    SELECT
        customer_id,
        DENSE_RANK() OVER (
            ORDER BY DATE(departure_date), origin, destination
        ) AS new_flight_group_id
    FROM flights_raw
)
UPDATE flights_raw f
SET flight_id = r.new_flight_group_id
FROM ranked r
WHERE f.customer_id = r.customer_id;
""") # with ranked as тут создаем временную таблицу для запроса
# dense_ rank присвоит номер каждой уникальной комбинации (по которым будем группировать строки где совпадают заданные параметры, не уточняю их, на случай если хотим их покрутить разные)
# DATE(departure_date) - тут по дню смотрим (не вплоть до секунд, иначе там вообще все рейсы уникальный айди)

In [59]:
df_sql = pd.read_sql("""
SELECT *
FROM flights_raw
ORDER BY flight_id ASC
""", conn)

df_sql.tail(5)

C:\Users\DELL\AppData\Local\Temp\ipykernel_17272\924622530.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql = pd.read_sql("""


,customer_id,departure_city,arrival_city,departure_date,flight_duration,delay_minutes,name,booking_class,frequent_flyer_status,route,ticket_price,competitor_price,demand,origin,destination,profitability,loyalty_points,churned,flight_id
95,2965,Lake Gracefurt,Jamesberg,2023-06-13 20:53:09,-0.94299,72,Christina Sanchez,Economy,Gold,BNE-SYD,318.903167,286.301632,-0.520139,MEL,LAX,1.129291,284,True,88
96,6107,Stevenville,Nicholasberg,2023-06-14 12:56:29,0.16641,37,Paul Morton,Business,Platinum,MEL-BNE,458.580341,288.079241,-1.199741,BNE,LHR,1.428864,560,False,89
97,4323,East Lori,Gallowaychester,2023-06-15 01:04:37,1.27581,63,Patrick Cook,Business,Platinum,MEL-BNE,304.892241,476.020503,-0.325967,SYD,LHR,0.874680,2180,False,90
98,3464,North Evelynland,Lucasstad,2023-06-16 21:25:28,-0.94299,9,Sarah Norton,Economy,Silver,MEL-BNE,292.562510,344.132375,1.227411,BNE,LHR,1.061499,2811,False,91
99,8855,Jonesside,Cookshire,2023-06-16 01:08:38,0.16641,28,Darren Garcia,Business,Platinum,SYD-MEL,375.903402,437.595545,-0.762854,SYD,SIN,0.838043,1746,False,92


In [73]:
# создадим и заполним 3 таблицы:

cur.execute("""
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    name TEXT,
    frequent_flyer_status TEXT,
    loyalty_points INT,
    churned BOOLEAN,
    flight_id INT
);

""")

cur.execute("""
DROP TABLE IF EXISTS flights;

CREATE TABLE flights (
    flight_id INT,
    departure_city TEXT,
    arrival_city TEXT,
    departure_date TIMESTAMP,
    flight_duration FLOAT,
    delay_minutes INT,
    route TEXT,
    origin TEXT,
    destination TEXT,
    demand FLOAT,
    profitability FLOAT
);
""")

cur.execute("""
DROP TABLE IF EXISTS bookings;

CREATE TABLE bookings (
    booking_id SERIAL PRIMARY KEY,
    customer_id INT,
    booking_class TEXT,
    ticket_price FLOAT,
    competitor_price FLOAT,
    flight_id INT
);
""")

In [74]:
cur.execute("""
INSERT INTO flights (
    flight_id,
    departure_city,
    arrival_city,
    departure_date,
    flight_duration,
    delay_minutes,
    route,
    origin,
    destination,
    demand,
    profitability
)
SELECT DISTINCT
    flight_id,
    departure_city,
    arrival_city,
    departure_date,
    flight_duration,
    delay_minutes,
    route,
    origin,
    destination,
    demand,
    profitability
FROM flights_raw;
""")
# тут SELECT DISTINCT используем чтобы у нас оставались только уникальные рейсы

cur.execute("""
INSERT INTO customers (
    customer_id,
    name,
    frequent_flyer_status,
    loyalty_points,
    churned,
    flight_id
)
SELECT
    customer_id,
    name,
    frequent_flyer_status,
    loyalty_points,
    churned,
    flight_id
FROM flights_raw;
""")

cur.execute("""
INSERT INTO bookings (
    customer_id,
    booking_class,
    ticket_price,
    competitor_price,
    flight_id
)
SELECT
    customer_id,
    booking_class,
    ticket_price,
    competitor_price,
    flight_id
FROM flights_raw;
""")

In [75]:
cur.execute("""
SELECT
    c.customer_id,
    c.name,
    c.flight_id,
    b.booking_class,
    b.ticket_price,
    f.origin,
    f.destination,
    f.departure_date
FROM customers c
JOIN bookings b ON c.flight_id = b.flight_id
JOIN flights f ON b.flight_id = f.flight_id
LIMIT 10;
""")
print(cur.fetchall()) # ну и тут joinы как будто совсем по фану стоят, мы сначала разделились на три таблицы, потом по ним же соединились

[(8457, 'Eric Lloyd', 20, 'Business', 475.267275684934, 'BNE', 'LHR', datetime.datetime(2023, 5, 1, 1, 9, 21)), (4756, 'Timothy Lloyd', 21, 'Economy', 206.6343348000344, 'BNE', 'SIN', datetime.datetime(2023, 5, 1, 21, 8, 9)), (3099, 'Eric Johnston', 22, 'First', 254.1411727260279, 'MEL', 'LHR', datetime.datetime(2023, 5, 1, 18, 34, 25)), (3769, 'Daniel Oliver', 23, 'Business', 370.6381276565254, 'MEL', 'LHR', datetime.datetime(2023, 5, 2, 20, 11, 9)), (9375, 'Monica Montgomery', 24, 'First', 457.5739219428346, 'MEL', 'LHR', datetime.datetime(2023, 5, 3, 19, 21, 8)), (1291, 'Anna Suarez', 25, 'First', 448.0751213503022, 'SYD', 'LHR', datetime.datetime(2023, 5, 3, 4, 40, 54)), (5777, 'Theodore Chang PhD', 26, 'First', 290.92446276729174, 'MEL', 'SIN', datetime.datetime(2023, 5, 5, 15, 27, 8)), (2863, 'Shawn Miller', 27, 'First', 411.6825198108389, 'SYD', 'LAX', datetime.datetime(2023, 5, 5, 2, 3, 26)), (9481, 'Hayden Wright', 28, 'Business', 161.10734008680674, 'BNE', 'LAX', datetime.dat

In [ ]:
cur.execute("SELECT COUNT(*) FROM flights_raw;")
print("flights_raw:", cur.fetchone())

cur.execute("SELECT COUNT(*) FROM flights;")
print("flights:", cur.fetchone())

cur.execute("SELECT COUNT(*) FROM customers;")
print("customers:", cur.fetchone())

cur.execute("SELECT COUNT(*) FROM bookings;")
print("bookings:", cur.fetchone()) # вроде строчки нигде не потеряли

flights_raw: (100,)
flights: (100,)
customers: (100,)
bookings: (100,)
